# Traffic Data Analysis

This notebook explores hourly vehicle counts from a public traffic dataset. It includes data checks, time-based analysis, visualizations, and automatically calculated observations.

**Dataset:** [Traffic Prediction Dataset on Kaggle](https://www.kaggle.com/datasets/fedesoriano/traffic-prediction-dataset)

**Before running:** Download the dataset CSV from Kaggle and put it in this folder. The notebook is designed for columns like `DateTime`, `Junction`, and `Vehicles`.

## 1. Import libraries

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load the dataset

Place the downloaded CSV in the same folder as this notebook.

In [ ]:
csv_files = list(Path('.').glob('*.csv'))
print('CSV files found:', [p.name for p in csv_files])

# If needed, set this to your downloaded filename, e.g. 'traffic.csv'
CSV_PATH = None
if CSV_PATH is None:
    if not csv_files:
        raise FileNotFoundError('No CSV found. Download the Kaggle dataset CSV and place it beside this notebook.')
    CSV_PATH = str(csv_files[0])

df = pd.read_csv(CSV_PATH)
print('Dataset shape:', df.shape)
display(df.head())
print('Columns:', df.columns.tolist())

## 3. Clean data and identify columns

In [ ]:
df.columns = [str(c).strip() for c in df.columns]

def find_column(possible_names, columns):
    lookup = {str(c).strip().lower().replace(' ', '').replace('_', ''): c for c in columns}
    for name in possible_names:
        key = name.lower().replace(' ', '').replace('_', '')
        if key in lookup:
            return lookup[key]
    return None

datetime_col = find_column(['DateTime', 'Date Time', 'timestamp', 'datetime', 'date'], df.columns)
junction_col = find_column(['Junction', 'Location', 'Road', 'site'], df.columns)
vehicles_col = find_column(['Vehicles', 'Vehicle Count', 'Traffic Volume', 'Count', 'traffic'], df.columns)
print('Date/time column:', datetime_col)
print('Junction/location column:', junction_col)
print('Vehicle-count column:', vehicles_col)

if datetime_col is None or vehicles_col is None:
    raise ValueError('Could not identify date/time and vehicle-count columns. Check the printed names and update the column lists.')

df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
df[vehicles_col] = pd.to_numeric(df[vehicles_col], errors='coerce')
df = df.dropna(subset=[datetime_col, vehicles_col]).copy()
df['hour'] = df[datetime_col].dt.hour
df['day_of_week'] = df[datetime_col].dt.day_name()
df['date'] = df[datetime_col].dt.date
print('Shape after removing invalid date/count rows:', df.shape)
display(df.head())

## 4. Data quality checks

In [ ]:
print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print('Duplicate rows:', int(df.duplicated().sum()))
print('Vehicle count summary:')
display(df[vehicles_col].describe().to_frame('value'))

## 5. Overall distribution of vehicle counts

In [ ]:
ax = df[vehicles_col].plot(kind='hist', bins=25, edgecolor='black')
ax.set_title('Distribution of Vehicle Counts')
ax.set_xlabel('Vehicles')
ax.set_ylabel('Number of observations')
plt.tight_layout()
plt.show()

## 6. Average traffic by hour

In [ ]:
hourly = df.groupby('hour')[vehicles_col].mean().reindex(range(24))
ax = hourly.plot(kind='line', marker='o')
ax.set_title('Average Vehicle Count by Hour')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Average vehicles')
ax.set_xticks(range(24))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Hour with highest average count:', int(hourly.idxmax()), 'average:', round(float(hourly.max()), 2))
print('Hour with lowest average count:', int(hourly.idxmin()), 'average:', round(float(hourly.min()), 2))

## 7. Average traffic by day of week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df.groupby('day_of_week')[vehicles_col].mean().reindex(day_order)
ax = daily.plot(kind='bar')
ax.set_title('Average Vehicle Count by Day of Week')
ax.set_xlabel('Day of week')
ax.set_ylabel('Average vehicles')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()
display(daily.to_frame('average_vehicles').round(2))

## 8. Compare junctions or locations

In [ ]:
if junction_col is not None:
    junction_summary = df.groupby(junction_col)[vehicles_col].agg(['count', 'mean', 'median', 'max']).sort_values('mean', ascending=False)
    display(junction_summary.round(2))
    ax = junction_summary['mean'].plot(kind='bar')
    ax.set_title('Average Vehicle Count by Junction/Location')
    ax.set_xlabel('Junction / location')
    ax.set_ylabel('Average vehicles')
    plt.tight_layout()
    plt.show()
else:
    print('No junction/location column detected; skipping this comparison.')

## 9. Average traffic over time

In [ ]:
time_series = df.groupby(df[datetime_col].dt.date)[vehicles_col].mean()
ax = time_series.plot(kind='line')
ax.set_title('Average Vehicle Count Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Average vehicles')
plt.tight_layout()
plt.show()

## 10. Automatically calculated observations

Review the outputs before using them in a report or portfolio.

In [ ]:
print(f'Cleaned observations: {len(df):,}')
print(f'Average vehicle count: {df[vehicles_col].mean():.2f}')
print(f'Median vehicle count: {df[vehicles_col].median():.2f}')
print(f'Maximum vehicle count: {df[vehicles_col].max():.0f}')
print(f'Minimum vehicle count: {df[vehicles_col].min():.0f}')
print(f'Hour with highest average count: {int(hourly.idxmax()):02d}:00')
if junction_col is not None:
    busiest = df.groupby(junction_col)[vehicles_col].mean().idxmax()
    print(f'Location with highest average count in this dataset: {busiest}')
print('\nThese are patterns in this dataset, not proof of what causes congestion.')

## Conclusion

After running the notebook, summarize the actual results here. Mention the busiest observed hours, days, or locations only when supported by the outputs. Note limitations such as the dataset's locations, time period, and the fact that vehicle counts alone do not explain why congestion occurs.

## Tools used
- Python
- Pandas
- Matplotlib
- Jupyter Notebook